# Adapters: a domain's words over the general machinery

`axiom` refuses domain vocabulary everywhere except here. Gate 3 walks every identifier in
`src/axiom` and fails on `channel`, `spend`, `geo`, `kpi`, `roas` — and it exempts exactly
one package, this one. That rule is what keeps a general toolkit general, and an **adapter**
is the seam it leaves open: the place where a domain says *plot* and *yield* and *nitrogen
rate*, and the machinery underneath keeps saying unit and outcome and dose.

This notebook builds one. `axiom.adapters.agronomy` is a second worked domain — a fertilizer
trial — written to be read as a template, and every section below is a decision you will
have to make again for your own.

**An adapter is five things and nothing else:**

| | | |
|---|---|---|
| **1** | a vocabulary | aliases, base dimensions, entity constructors |
| **2** | a roles spec | which columns play which part, and a `RoleMap` translation |
| **3** | a loader | the domain's file shape → a `Panel` |
| **4** | a preset | the `SurfaceSpec` a practitioner would have written, with the defaults argued |
| **5** | domain quantities | the numbers the domain asks for, assembled from `estimands` |

**And an adapter is not:** new mathematics. If you find yourself writing a solver, a
likelihood, or a transform, it belongs *below* — in `axiom.surface`, `axiom.estimands`,
`axiom.design` — and under a general name. Section 4 is a worked example of exactly this
temptation, resisted.

In [ ]:
import numpy as np
import pandas as pd

from axiom.adapters import agronomy, marketing
from axiom.core import D, Unsupported, dimensionless
from axiom.estimands import EstimandResult
from axiom.surface import fit

print("adapters ship two domains:", agronomy.__name__.split(".")[-1],
      "and", marketing.__name__.split(".")[-1])
print("agronomy's public surface:", len(agronomy.__all__), "names")

## 1 · The vocabulary

Three moves, in increasing order of commitment.

**Aliases** cost nothing and are the whole point of the exercise: a plot *is* a `Unit`, a
nutrient *is* a `Treatment`. Writing `Plot = Unit` means a reader of the adapter sees their
own word and a reader of the core sees theirs, and there is no translation layer to drift.

**Base dimensions** are the one piece of global state a domain adds. `BASES.declare` is
idempotent, so declaring `mass` in an adapter is safe even if a notebook, a test, or another
adapter declared it first — and `axiom.io` records any non-default declaration in the
analysis manifest and re-declares it on load, so a saved analysis still type-checks in a
fresh process.

In [ ]:
print("Plot     is Unit     :", agronomy.Plot.__name__)
print("Nutrient is Treatment:", agronomy.Nutrient.__name__)
print("Yield    is Outcome  :", agronomy.Yield.__name__)
print()
print("mass          ", agronomy.MASS)
print("area          ", agronomy.AREA)
print("mass per area ", agronomy.MASS_PER_AREA, " <- both the rate and the yield")
print()
print("declare is idempotent:", agronomy.MASS is not D.mass, "(different objects)",
      "| equal:", agronomy.MASS == D.mass)

**Entity constructors** are where the adapter spends its judgement. `nutrient_rate` does not
just fill in a dimension — it decides that a fertilizer dose is a mass per unit area, that
its unit string is `kg/ha`, and that a price makes it a costed dose with a numeraire. A user
of the adapter never has to be right about any of that.

Note what `soil_test` refuses to do. Soil organic carbon is a percentage, pH is a logarithm,
and residual N is a rate — so the constructor declares them **dimensionless** rather than
guessing, because a covariate that claims a dimension it cannot support fails gate 10 later
and in a much less obvious place.

In [ ]:
rate = agronomy.nutrient_rate("nitrogen", price=2.20)
print("dose     ", rate.name, "|", rate.dimension, "|", rate.unit, "| costed in", rate.numeraire)
free = agronomy.nutrient_rate("phosphorus")
print("uncosted ", free.name, "|", free.numeraire, " <- no price, no numeraire, no pretence")

carbon = agronomy.soil_test("soil_carbon", unit="%")
print("covariate", carbon.name, "|", carbon.dimension, "|", carbon.unit)
print("dimensionless?", carbon.dimension == dimensionless())

try:
    agronomy.nutrient_rate("nitrogen", price=-1.0)
except ValueError as e:
    print("\nand a bad price is a ValueError, not a silent zero:", e)

## 2 · Roles, and 3 · the loader

Every adapter needs a `Spec` that says which columns play which part, and a function that
turns it into the general `RoleMap`. This is the least interesting code in the package and
the most load-bearing: it is the only place the domain's column names exist, so everything
downstream is addressed by role.

The loader is thin on purpose. `panel_from_trial` selects, validates, and hands off — it
does not impute, and an unharvested plot stays a missing cell that `Panel.completeness`
will report. The marketing adapter's `panel_from_mff` is the same function with a pivot in
front of it, because the domain's file format is the adapter's problem and nobody else's.

In [ ]:
LEVELS = (0.0, 45.0, 90.0, 135.0, 180.0, 225.0, 270.0)


def a_trial(seed: int = 0) -> pd.DataFrame:
    # Twenty-eight plots, three seasons, seven nitrogen rates rotated within each plot so
    # that the response is identified *within* a plot and not across them.
    rng = np.random.default_rng(seed)
    rows = []
    for plot in range(28):
        for season in range(3):
            n = float(LEVELS[(plot + 2 * season) % len(LEVELS)])
            rows.append({
                "plot": f"p{plot:02d}",
                "season": season,
                "grain": 3.0 + 5.2 * (1.0 - np.exp(-n / 70.0)) + rng.normal(0.0, 0.30),
                "nitrogen": n,
                "soil_carbon": float(rng.normal(1.4, 0.2)),
                "agronomist_notes": "not a column any role names",
            })
    return pd.DataFrame(rows)


roles = agronomy.TrialRoles(
    harvest="grain", nutrients=("nitrogen",), plot="plot", season="season",
    soil_tests=("soil_carbon",),
)
mapped = agronomy.role_map(roles)
print("roles name these columns:", roles.columns)
print("outcome  ->", mapped.outcome[1].name, mapped.outcome[1].dimension, mapped.outcome[1].unit)
print("treatment->", {k: (str(v.dimension), v.unit) for k, v in mapped.treatments.items()})

panel = agronomy.panel_from_trial(a_trial(), roles)
print("\npanel:", panel.completeness())
print("the un-named column was dropped:", "agronomist_notes" not in panel.frame.columns)

## 4 · The preset, and the kernel that already existed

Here is the temptation. The classical nitrogen-response curve is **Mitscherlich's**,
$Y = A\,(1 - e^{-kN})$: yield rises to an asymptote, each extra kilogram doing less than the
last. It has a name, a century of literature, and an obvious home in an agronomy adapter.

It is already in `axiom.surface`, called `ExponentialKernel`, because "saturating, concave
everywhere, one scale parameter" is not an agronomic idea — it is a shape. The adapter's job
was to *find* it and make it the default, not to add it.

That is the test to apply every time: **would this thing make sense to someone in another
field?** If yes it goes below, under a general name. If genuinely not — a domain's file
format, a domain's pricing convention — it stays here.

In [ ]:
spec = agronomy.trial_spec(panel, amplitude_scale=5.0)
spec = spec.model_copy(update={"intercept_scale": 4.0, "noise_scale": 1.0})
print("kernel chosen:", {k: v.name for k, v in spec.kernels.items()})
print("reference dose (mean applied rate):",
      {k: round(v.reference_dose, 1) for k, v in spec.kernels.items()})
print("intercept:", spec.intercept, "| carryover:", {k: v.name for k, v in spec.carryover.items()})
print("\nthe quadratic an agronomist fits when yield may *fall* at high rates:")
bendy = agronomy.trial_spec(panel, kernel="polynomial", name="quadratic_response")
print("  ", {k: v.name for k, v in bendy.kernels.items()},
      "-- no saturating family can bend back down")

Two defaults in `trial_spec` are arguments, not conventions, and the docstring makes both.

**The intercept is `"shared"`, not `"hierarchical"`.** Plots differ — that is why fields are
blocked — so hierarchical is the better description. But a field trial is many plots and few
seasons, and a hierarchical intercept over twenty-eight plots with three harvests each is
thin enough that the Laplace mode search will not certify its own Hessian. It says so, as a
typed value, rather than returning numbers.

**Residual carryover is off.** Unused nitrogen really is in the soil next season, so the
confound is real whether or not it is modelled — but it is only *identified* when the applied
rate changes within a plot, which is a property of the trial, not of the crop.

In [ ]:
thin = agronomy.trial_spec(panel, intercept="hierarchical", amplitude_scale=5.0)
thin = thin.model_copy(update={"intercept_scale": 4.0, "noise_scale": 1.0})
attempt = fit(thin, panel, backend="laplace", draws=200, seed=5)
print("hierarchical on three seasons ->", type(attempt.posterior).__name__)
print("  ", getattr(attempt.posterior, "reason", "")[:150])
print("\nand nothing downstream pretends otherwise:")
print("  ", type(agronomy.response_to(attempt, "nitrogen")).__name__)

result = fit(spec, panel, backend="laplace", draws=800, seed=5)
print("\nshared ->", type(result.posterior).__name__)

## 5 · Domain quantities, and the three facets the adapter decides

An agronomist asks for four numbers. All four are `estimands.Estimand` objects realized
against the fit — the adapter writes no arithmetic — and the whole of its contribution is
**choosing the facets**, which is exactly where a domain's meaning lives.

| facet | marketing chose | agronomy chooses | because |
|---|---|---|---|
| `level.unit` | `aggregate` (sum over geos) | `individual` (mean over plots) | revenue is extensive; a yield in t/ha is **intensive**, and summing twenty-eight of them gives twenty-eight times the answer |
| `window.basis` | `cumulative` | `per_period` | three harvests of four tonnes a hectare is four tonnes a hectare a **year**, not twelve |
| `reference` | zero spend | zero rate | an unfertilized plot is a real, run, observed thing — which is why trials include one |

Get these wrong and every number is off by a factor that no test will catch, because the
units still agree. Getting them right, once, in the adapter, is the entire reason the adapter
exists.

In [ ]:
for label, quantity in (
    ("yield response", agronomy.response_to),
    ("agronomic efficiency", agronomy.agronomic_efficiency),
    ("marginal product", agronomy.marginal_product),
    ("N elasticity", agronomy.nutrient_elasticity),
):
    out = quantity(result, "nitrogen", seed=1)
    assert isinstance(out, EstimandResult)
    s = out.summary
    print(f"{label:22s} {s.mean:8.4f}  [{s.interval.lower:7.4f}, {s.interval.upper:7.4f}]  "
          f"dim {str(out.dimension):8s} unit {out.unit}")
print("\nevery one carries its interval definition, its mass, and its assumptions:")
one = agronomy.response_to(result, "nitrogen", seed=1)
assert isinstance(one, EstimandResult)
print("  status:", one.status, "| interval:", one.summary.interval.definition,
      "at", one.summary.interval.mass)
print("  assumes:", [a.name for a in one.assumptions])

Read the second row. **Agronomic efficiency is dimensionless** — kilograms of grain per
kilogram of nitrogen — because a rate and a yield are both a mass per unit area and `axiom`
has no notion of *substance*. That is not a bug to work around; it is the convention the
field already uses, and it is what makes the number comparable across crops and countries.
The unit string `t/ha/kg/ha` survives to say which two masses were involved.

Compare the marketing adapter, where the same `ratio` estimand is `roas` and is dimensionless
**only** when the outcome is currency-valued — so `roas` checks, and returns `Unsupported`
for a count-valued outcome rather than a mislabelled number. Same estimand, opposite
precondition, because the domains put different things on the two sides.

In [ ]:
print("marketing.roas guards its dimension:")
print(" ", marketing.roas.__doc__.strip().splitlines()[0])
print("\nagronomy.agronomic_efficiency does not need to:")
print(" ", agronomy.agronomic_efficiency.__doc__.strip().splitlines()[0])

## 6 · The quantity `estimands` cannot express

Every `Estimand` is a functional of the response **at a stated intervention**. The number an
agronomist actually wants is the other way round: the *rate at which* a functional takes a
stated value — where the marginal product falls to the price ratio, and the last kilogram of
nitrogen stops paying for itself. That is an inversion, and no facet expresses it.

So it does not get forced into an `Estimand`. It is its own `Spec`, and being its own spec is
what lets it carry the things it needs: the prices, the dose range it was searched in, how the
interval was constructed, and whether the trial bracketed the answer at all.

The interval is by **inversion** — the rates at which the marginal-product band still contains
the price ratio — and the spec says so in `detail` rather than letting a reader assume it is
a symmetric error bar.

In [ ]:
prices = agronomy.Prices(harvest=220.0, nutrient=2.20)
print(f"grain {prices.harvest} {prices.currency}/t, N {prices.nutrient} {prices.currency}/kg")
print(f"-> break-even marginal product = {prices.ratio:.5f} t/ha per kg/ha\n")

optimum = agronomy.economic_optimum(result, "nitrogen", prices, seed=2)
assert isinstance(optimum, agronomy.EconomicOptimum)
print(f"economic optimum rate   {optimum.rate:6.1f} kg N/ha "
      f"[{optimum.lower:.1f}, {optimum.upper:.1f}]  ({optimum.mass:.0%} {optimum.definition},"
      f" by inversion)")
print(f"yield gain there        {optimum.expected_gain:6.3f} t/ha")
print(f"partial budget          {optimum.profit_over_zero:6,.0f} {prices.currency}/ha")
print(f"searched                {optimum.searched[0]:.0f} to {optimum.searched[1]:.0f} kg/ha,"
      f" bracketed={optimum.bracketed}")

And now the case that matters more. Make nitrogen cheap enough and the optimum moves past the
highest rate anybody applied. There is a number the arithmetic would happily produce — the
curve extrapolates — and it would be a fabrication.

In [ ]:
for label, prices in (
    ("N at 2.20/kg", agronomy.Prices(harvest=220.0, nutrient=2.20)),
    ("N at 1.10/kg", agronomy.Prices(harvest=220.0, nutrient=1.10)),
    ("N at 0.20/kg", agronomy.Prices(harvest=220.0, nutrient=0.20)),
):
    got = agronomy.economic_optimum(result, "nitrogen", prices, seed=2)
    if isinstance(got, Unsupported):
        print(f"{label}: {type(got).__name__}")
        print(f"           {got.reason}")
        print(f"           missing={got.missing} detail={got.detail}")
    else:
        print(f"{label}: {got.rate:6.1f} kg N/ha  [{got.lower:.0f}, {got.upper:.0f}]")

`Unsupported` is not an error and not a `None`. It is a typed value that names what is
missing (`dose_range`), says what was searched, and tells the caller what to do about it —
apply more nitrogen, or report the range. Rule 5 of the repo exists because four wrong-number
bugs in the parent traced to code that swallowed this situation instead.

## 7 · What goes where

The one judgement you will make repeatedly, in a table.

| you are writing | it goes | test |
|---|---|---|
| "a plot is a unit" | **adapter** | is it a translation? |
| a domain's file format | **adapter** | would another field ever read this file? |
| which facets a domain's number implies | **adapter** | does the domain disagree with the default? |
| a response shape | `axiom.surface` | would another field recognize the curve? |
| a new functional of the response | `axiom.estimands` | is it a functional at a stated intervention? |
| a design criterion | `axiom.design` | is it about what data to collect? |
| one study's synthetic world | **`nbs/case-studies/`** | is it *this* dataset, or any dataset of this shape? |

The last row is the one people get wrong in the other direction. `nbs/case-studies/rutherford/scattering.py`
is a hand-built `SupportsForward`, a base dimension, and a dozen domain functions — every
ingredient of an adapter — and it is correctly *not* one, because it describes one experiment
rather than a class of them. An adapter earns its place when the second user shows up.

## What this notebook demonstrated

* An adapter is **five parts**: vocabulary, roles, loader, preset, domain quantities — and no
  new mathematics.
* Gate 3 exempts `adapters/` and nothing else, which is what makes the exemption safe.
* Mitscherlich's equation was already in the library as `ExponentialKernel`. **Look before
  you add.**
* The adapter's real contribution is **choosing the facets** — `level`, `basis`, `reference` —
  because a yield is intensive and revenue is extensive, and no unit check will catch that.
* A quantity `estimands` cannot express gets its own `Spec`, carries its own provenance, and
  returns `Unsupported` rather than extrapolating past the data.